# QASPER Phase 4C · Qwen 7B feasibility

Use a **free T4 runtime**. If the default runtime has Python 3.13, select runtime version **2026.07** (Python 3.12) if available. Upload only the exported packet and, when resuming, a previous checkpoint. No API key, Drive mount, external serving, or paid runtime is required. Stop if the GPU or CUDA preflight fails. Run one stage at a time and download checkpoints before ending the session.

In [ ]:
"""Standalone, standard-library checkpoint protocol used by the Colab notebook."""

import hashlib
import json
import math
from pathlib import Path


STAGES = ("sanity", "original", "confirmation")
RUNTIME_ID = "llama-cpp-python-0.3.16-cu124-wheel-v1"
WHEEL_BASE = "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.16-cu124/"
WHEELS = {
    "cp311": ("llama_cpp_python-0.3.16-cp311-cp311-linux_x86_64.whl",
              "a455263e7608a51c26079e82fe524f9b7f1f73ee9fc563221a297de8da2a8c51", 551261692),
    "cp312": ("llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl",
              "431fd9e68ff699bfa9ae6a01b05a0b6ddc0de97aa63c503e84141ae408502a11", 551261698),
}


def select_wheel(system, machine, python_version):
    abi = f"cp{python_version[0]}{python_version[1]}"
    if system != "Linux" or machine != "x86_64" or abi not in WHEELS:
        raise ValueError("Use a free Linux x86_64 Colab GPU runtime with Python 3.11 or 3.12")
    return WHEELS[abi]


def digest(value):
    data = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_hash(path):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save(path, value):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2) + "\n",
                         encoding="utf-8")
    temporary.replace(path)


def render(chat):
    return "".join(f"<|im_start|>{turn['role']}\n{turn['content']}<|im_end|>\n" for turn in chat) + (
        "<|im_start|>assistant\n")


def validate_packet(packet):
    payload = packet["payload"]
    if digest(payload) != packet["sha256"] or payload["config"]["protocol"] != "phase4c-v1":
        raise ValueError("Packet fingerprint or protocol mismatch")
    cases = payload["cases"]
    if (len({case["id"] for case in cases}) != len(cases)
            or [case["stage"] for case in cases] != sorted(
                (case["stage"] for case in cases), key=STAGES.index)
            or {stage: sum(c["stage"] == stage for c in cases) for stage in STAGES}
            != {"sanity": 3, "original": 8, "confirmation": 15}):
        raise ValueError("Invalid case manifest")
    return payload


def validate_checkpoint(packet, checkpoint):
    payload = validate_packet(packet)
    if checkpoint["packet_sha256"] != packet["sha256"]:
        raise ValueError("Checkpoint belongs to a different packet")
    cases = payload["cases"]
    rows = checkpoint["rows"]
    config = payload["config"]
    environment = None
    for session in checkpoint["sessions"]:
        if session.get("runtime_id") != RUNTIME_ID:
            raise ValueError("Use a separate checkpoint for this binary runtime")
        wheel = session["wheel"]
        gpu = session["gpu_before"]
        probe = session["gpu_probe"]
        if ((wheel["name"], wheel["sha256"], wheel["bytes"]) not in WHEELS.values()
                or session["model_revision"] != config["revision"] or session["weight_shards"] != config["shards"]
                or session["versions"]["llama-cpp-python"] != config["backend_version"]
                or "T4" not in gpu["name"] or gpu["compute_capability"] != "7.5"
                or gpu["free_mib"] < config["minimum_free_gpu_mib"]
                or session["cuda_driver_api"] < 12040
                or probe["process_used_mib"] < 3072 or probe["tokens_evaluated"] < 1
                or session["settings"] != {"n_gpu_layers": -1, "n_batch": 256, "n_ctx": config["context_window"]}):
            raise ValueError("Runtime session differs from the frozen configuration")
        key = (wheel["sha256"], gpu["name"], gpu["driver"], session["cuda_driver_api"], session["versions"])
        if environment is not None and key != environment:
            raise ValueError("Do not combine records from different runtime environments")
        environment = key
    if len(rows) > len(cases):
        raise ValueError("Too many result records")
    for case, wrapped in zip(cases, rows):
        row = wrapped["record"]
        if (digest(row) != wrapped["sha256"] or row["id"] != case["id"]
                or row["stage"] != case["stage"] or row["prompt_sha256"] != digest(render(case["chat"]))):
            raise ValueError(f"Invalid result record for {case['id']}")
        if not 0 < row["prompt_tokens"] <= payload["config"]["context_window"] - payload["config"]["max_output"]:
            raise ValueError("Invalid prompt token count")
        if not 0 <= row["output_tokens"] <= payload["config"]["max_output"]:
            raise ValueError("Invalid output token count")
        if not math.isfinite(row["latency_s"]) or row["latency_s"] <= 0:
            raise ValueError("Invalid measured latency")
        if type(row["session"]) is not int or not 0 <= row["session"] < len(checkpoint["sessions"]):
            raise ValueError("Invalid result session")
    for stage in ("original", "confirmation"):
        if any(row["record"]["stage"] == stage for row in rows) and stage not in checkpoint["approvals"]:
            raise ValueError(f"Missing manual stage approval: {stage}")
    return rows


def append_result(packet, checkpoint, path, row):
    candidate = checkpoint | {"rows": [*checkpoint["rows"], {"record": row, "sha256": digest(row)}]}
    validate_checkpoint(packet, candidate)
    save(path, candidate)
    checkpoint["rows"] = candidate["rows"]

def parse_answer(text, passage_ids):
    result = {"status": "malformed", "abstained": None, "claims": [], "answer": "",
              "invalid_ids": [], "missing_citations": 0, "claim_count": 0}
    try:
        obj = json.loads(text)
    except (ValueError, TypeError):
        return result
    if (not isinstance(obj, dict) or set(obj) != {"abstain", "claims"}
            or type(obj["abstain"]) is not bool or not isinstance(obj["claims"], list)):
        return result
    claims = obj["claims"]
    if obj["abstain"]:
        if not claims:
            result.update(status="abstained", abstained=True)
        return result
    if not 1 <= len(claims) <= 2:
        return result
    cleaned = []
    for claim in claims:
        if (not isinstance(claim, dict) or set(claim) != {"text", "citations"}
                or not isinstance(claim["text"], str) or not claim["text"].strip()
                or not isinstance(claim["citations"], list) or len(claim["citations"]) > 5
                or any(not isinstance(c, str) for c in claim["citations"])):
            return result
        cited = claim["citations"]
        valid = list(dict.fromkeys(cid for cid in cited if cid in passage_ids))
        result["invalid_ids"].extend(cid for cid in cited if cid not in passage_ids)
        result["missing_citations"] += int(not cited)
        cleaned.append({"text": claim["text"], "citations": valid})
    status = "citation_error" if result["invalid_ids"] or result["missing_citations"] else "answered"
    result.update(status=status, abstained=False, claims=cleaned, claim_count=len(cleaned),
                  answer=" ".join(claim["text"] for claim in cleaned))
    return result


In [ ]:
from google.colab import files
import ctypes, os, platform, resource, shutil, subprocess, sys, threading, time
from importlib.metadata import version

ROOT = Path('/content/phase4c')
ROOT.mkdir(exist_ok=True)
print('Upload phase4c-packet.json')
uploaded = files.upload()
if set(uploaded) != {'phase4c-packet.json'}:
    raise ValueError('Upload only phase4c-packet.json')
packet = json.loads(uploaded['phase4c-packet.json'])
data = validate_packet(packet)
cfg = data['config']
checkpoint_path = ROOT / 'phase4c-checkpoint.json'
checkpoint = {'packet_sha256': packet['sha256'], 'sessions': [], 'approvals': {}, 'rows': []}
if checkpoint_path.exists():
    checkpoint = json.loads(checkpoint_path.read_text(encoding='utf-8'))
validate_checkpoint(packet, checkpoint)
print(packet['sha256'], [(s, sum(c['stage'] == s for c in data['cases'])) for s in STAGES])

In [ ]:
# Optional resume after a Colab reset. Set True only to upload a previously downloaded checkpoint.
RESUME = False
if RESUME:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload a single checkpoint JSON')
    resumed = json.loads(next(iter(uploaded.values())))
    validate_checkpoint(packet, resumed)
    if resumed['rows'][:len(checkpoint['rows'])] != checkpoint['rows']:
        raise ValueError('The uploaded checkpoint does not extend local progress')
    checkpoint = resumed
    save(checkpoint_path, checkpoint)

In [ ]:
def cmd(*args):
    return subprocess.check_output(args, text=True, timeout=15).strip()

def gpu():
    line = cmd('nvidia-smi', '--query-gpu=name,memory.total,memory.free,memory.used,driver_version,compute_cap',
               '--format=csv,noheader,nounits').splitlines()[0]
    name, total, free, used, driver, capability = (part.strip() for part in line.split(','))
    return {'name': name, 'total_mib': int(total), 'free_mib': int(free),
            'used_mib': int(used), 'driver': driver, 'compute_capability': capability}

def process_gpu_mib():
    lines = cmd('nvidia-smi', '--query-compute-apps=pid,used_gpu_memory',
                '--format=csv,noheader,nounits').splitlines()
    pids = {os.getpid()}
    for line in Path('/proc/self/status').read_text().splitlines():
        if line.startswith('NSpid:'):
            pids.update(map(int, line.split()[1:]))
    allocations = []
    for line in lines:
        fields = [field.strip() for field in line.split(',')]
        if len(fields) == 2 and fields[0].isdigit() and int(fields[0]) in pids and fields[1].isdigit():
            allocations.append(int(fields[1]))
    return sum(allocations)

wheel_name, wheel_sha, wheel_bytes = select_wheel(platform.system(), platform.machine(), sys.version_info)
preflight = gpu()
print(preflight)
if ('T4' not in preflight['name'] or preflight['compute_capability'] != '7.5'
        or preflight['free_mib'] < cfg['minimum_free_gpu_mib']):
    raise RuntimeError('Require a T4 with at least 8192 MiB free. Stop before downloading assets.')
driver = ctypes.CDLL('libcuda.so.1')
driver_api = ctypes.c_int()
if driver.cuInit(0) or driver.cuDriverGetVersion(ctypes.byref(driver_api)) or driver_api.value < 12040:
    raise RuntimeError('A working CUDA driver API at least 12.4 is required')
if shutil.disk_usage(ROOT).free < 8 * 2**30:
    raise RuntimeError('Insufficient runtime disk space')

# Install the exact upstream wheel. No index or source-build fallback for llama-cpp-python.
wheel_path = ROOT / wheel_name
started = time.perf_counter()
if 'llama_cpp' in sys.modules and globals().get('active_wheel_sha256') != wheel_sha:
    raise RuntimeError('Restart the Colab session to unload the previous backend')
if not wheel_path.exists():
    partial = wheel_path.with_suffix('.part')
    subprocess.run(['curl', '-fL', '--retry', '1', '--max-time', '360', '-C', '-',
                    WHEEL_BASE + wheel_name, '-o', str(partial)], check=True, timeout=370)
    if partial.stat().st_size != wheel_bytes or file_hash(partial) != wheel_sha:
        raise RuntimeError('Wheel size or SHA-256 mismatch')
    partial.replace(wheel_path)
if wheel_path.stat().st_size != wheel_bytes or file_hash(wheel_path) != wheel_sha:
    raise RuntimeError('Cached wheel size or SHA-256 mismatch')
if 'llama_cpp' not in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:',
                    'diskcache==5.6.3'], check=True, timeout=60)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--only-binary=:all:',
                    '--force-reinstall', '--no-deps', str(wheel_path)], check=True, timeout=120)
import llama_cpp
from llama_cpp import Llama, LlamaGrammar
if version('llama-cpp-python') != cfg['backend_version'] or not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError('Pinned backend or CUDA offload check failed; stop before downloading weights')
active_wheel_sha256 = wheel_sha
grammar_probe = LlamaGrammar.from_json_schema(json.dumps(data['schema']), verbose=False)
setup_s = time.perf_counter() - started
if gpu()['free_mib'] < cfg['minimum_free_gpu_mib']:
    raise RuntimeError('GPU memory changed after preflight')
print(f'CUDA wheel and response grammar ready in {setup_s:.1f}s')


In [ ]:
weights = ROOT / 'weights'
weights.mkdir(exist_ok=True)
for shard in cfg['shards']:
    destination = weights / shard['name']
    if not destination.exists():
        partial = destination.with_suffix('.part')
        url = f"https://huggingface.co/{cfg['model']}/resolve/{cfg['revision']}/{shard['name']}"
        subprocess.run(['curl', '-fL', '--retry', '2', '-C', '-', url, '-o', str(partial)], check=True)
        if file_hash(partial) != shard['sha256']:
            raise RuntimeError('Shard SHA-256 mismatch')
        partial.replace(destination)
    if file_hash(destination) != shard['sha256']:
        raise RuntimeError('Cached shard SHA-256 mismatch')

def measure(function):
    samples = [gpu()['used_mib']]
    finished = threading.Event()
    def poll():
        while not finished.wait(.5):
            samples.append(gpu()['used_mib'])
    monitor = threading.Thread(target=poll, daemon=True)
    monitor.start()
    started = time.perf_counter()
    try:
        result = function()
        elapsed = time.perf_counter() - started
    finally:
        finished.set()
        monitor.join()
    return result, elapsed, max(samples + [gpu()['used_mib']])

process_mib_before = process_gpu_mib()
model, load_s, load_peak = measure(lambda: Llama(
    model_path=str(weights / cfg['shards'][0]['name']), n_gpu_layers=-1,
    n_ctx=cfg['context_window'], n_batch=256, seed=cfg['seed'], verbose=True))
template = model.metadata.get('tokenizer.chat_template', '')
if model.metadata.get('general.architecture') != 'qwen2' or not all(
    tag in template for tag in ('<|im_start|>', '<|im_end|>')):
    raise RuntimeError('GGUF model or chat-template family mismatch')
# Forward-only CUDA check. This uses no evaluation input and generates no answer.
probe_tokens = model.tokenize(b'GPU compatibility check.', add_bos=False, special=False)
model.reset()
model.eval(probe_tokens)
process_mib_after = process_gpu_mib()
if process_mib_after - process_mib_before < 3072:
    raise RuntimeError('Could not verify GPU model offload for this Python process')
model.reset()
print('Verified T4 forward pass and process GPU allocation:', process_mib_after, 'MiB')
# Validate every frozen case before attempting inference. No repacking or silent truncation.
token_lists = {}
for case in data['cases']:
    tokens = model.tokenize(render(case['chat']).encode(), add_bos=False, special=True)
    empty = model.tokenize(render(case['empty_chat']).encode(), add_bos=False, special=True)
    if (len(tokens) + cfg['max_output'] > cfg['context_window']
            or len(tokens) - len(empty) > cfg['evidence_tokens']
            or len(case['context']) > cfg['max_passages']):
        raise RuntimeError(f"Budget exceeded for {case['id']}")
    token_lists[case['id']] = tokens

session = len(checkpoint['sessions'])
checkpoint['sessions'].append({
    'runtime_id': RUNTIME_ID, 'gpu_before': preflight, 'cuda_driver_api': driver_api.value, 'python': sys.version, 'platform': platform.platform(),
    'versions': {name: version(name) for name in ('llama-cpp-python', 'numpy', 'diskcache', 'jinja2')},
    'runtime_info': llama_cpp.llama_print_system_info().decode(errors='replace'),
    'wheel': {'name': wheel_name, 'sha256': wheel_sha, 'bytes': wheel_bytes},
    'wheel_setup_seconds': setup_s,
    'gpu_probe': {'process_used_mib': process_mib_after, 'process_before_mib': process_mib_before,
                  'tokens_evaluated': len(probe_tokens)},
    'model_revision': cfg['revision'],
    'weight_shards': cfg['shards'], 'chat_template_sha256': digest(template),
    'load_seconds': load_s, 'load_peak_device_used_mib': load_peak,
    'settings': {'n_gpu_layers': -1, 'n_batch': 256, 'n_ctx': cfg['context_window']},
})
validate_checkpoint(packet, checkpoint)
save(checkpoint_path, checkpoint)
print(checkpoint['sessions'][session])

## Run a stage

After review, set RUN_EVALUATION to True and run **sanity** first. Inspect the raw JSON. Require blue, a negative answer about collection after closing, and explicit abstention for the absent name. Both answers must cite P1 and all three records must have coherent fields.

Then set the stage to **original** and explicitly approve it. Compare those eight outputs with the local-only review template and packet passages. Require at least four clearly correct, supported answers, both native negatives declined, and valid structures and citation IDs before approving **confirmation**. Confirmation uses the pre-frozen 15 cases. Its final decision requires semantic review using the criteria in the local report. The notebook does not infer a semantic pass from JSON. Download the checkpoint after every stage, including failures, and before a runtime reset.

In [ ]:
RUN_EVALUATION = False  # Enable only after reviewing the notebook and GPU checks.
if not RUN_EVALUATION:
    raise RuntimeError('Evaluation is disabled pending review')

STAGE = 'sanity'  # Change only to 'original' or 'confirmation' after reviewing the previous stage.
APPROVAL = ''  # For later stages, type e.g. 'approve original' after completing the rubric above.
MAX_NEW = 4    # Repeat this cell to continue a long stage; download a checkpoint between batches.

def generate(case):
    tokens = token_lists[case['id']]
    model.reset()
    model.set_seed(cfg['seed'])
    grammar = LlamaGrammar.from_json_schema(json.dumps(data['schema']), verbose=False)
    response, elapsed, peak = measure(lambda: model.create_completion(
        tokens, max_tokens=cfg['max_output'], temperature=cfg['temperature'],
        seed=cfg['seed'], stream=False, grammar=grammar))
    if response['usage']['prompt_tokens'] != len(tokens):
        raise RuntimeError('Backend reported a different prompt length')
    choice = response['choices'][0]
    raw = choice['text']
    output_tokens = response['usage']['completion_tokens']
    return {
        'id': case['id'], 'stage': case['stage'], 'session': session, 'raw': raw,
        'prompt': render(case['chat']), 'prompt_sha256': digest(render(case['chat'])),
        'prompt_tokens': len(tokens), 'output_tokens': output_tokens,
        'latency_s': elapsed, 'finish_reason': choice['finish_reason'],
        'tokens_per_second_end_to_end': output_tokens / elapsed,
        'peak_device_used_mib_sampled': peak, 'peak_process_rss_kib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    }

stage_index = STAGES.index(STAGE)
validate_checkpoint(packet, checkpoint)
earlier = [c for c in data['cases'] if STAGES.index(c['stage']) < stage_index]
if len(checkpoint['rows']) < len(earlier):
    raise RuntimeError('Complete the previous stages first')
if STAGE != 'sanity' and STAGE not in checkpoint['approvals']:
    previous = STAGES[stage_index - 1]
    prior_cases = {c['id']: c for c in data['cases'] if c['stage'] == previous}
    parsed = [(prior_cases[w['record']['id']], parse_answer(w['record']['raw'], {
                  p['id'] for p in prior_cases[w['record']['id']]['context']}))
              for w in checkpoint['rows'] if w['record']['stage'] == previous]
    minimum_answers = 2 if previous == 'sanity' else 4
    if (any(result['status'] not in {'answered', 'abstained'} for _, result in parsed)
            or sum(c['answerable'] and r['status'] == 'answered' for c, r in parsed) < minimum_answers
            or any(not c['answerable'] and r['status'] != 'abstained' for c, r in parsed)):
        raise RuntimeError('Previous stage failed its automatic screen; download the checkpoint and stop')
    if APPROVAL != f'approve {STAGE}':
        raise RuntimeError('Manual stage approval required')
    checkpoint['approvals'][STAGE] = APPROVAL
    save(checkpoint_path, checkpoint)
done = {row['record']['id'] for row in checkpoint['rows']}
pending = [c for c in data['cases'] if c['stage'] == STAGE and c['id'] not in done]
for case in pending[:MAX_NEW]:
    row = generate(case)
    append_result(packet, checkpoint, checkpoint_path, row)
    print(case['id'], row['raw'], f"{row['latency_s']:.2f}s", flush=True)
print('Saved', len(checkpoint['rows']), 'of', len(data['cases']), 'records.')

In [ ]:
# Download at any time after initialization. Upload only the checkpoint on resume.
validate_checkpoint(packet, checkpoint)
save(checkpoint_path, checkpoint)
files.download(str(checkpoint_path))